# Setting up:

In [20]:
!pip install duckdb

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 24.6 MB/s  0:00:00m0:00:0100:01


In [1]:
# AWS CLI (to download from the public S3 bucket)
# See AWS docs if your distro differs
!curl "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o awscliv2.zip
!unzip -q awscliv2.zip && sudo ./aws/install

# DuckDB CLI (choose the latest release for your OS/arch)
# (Or `pip install duckdb` if you prefer Python)
!curl -L -o duckdb_cli.zip https://github.com/duckdb/duckdb/releases/download/v1.1.1/duckdb_cli-linux-amd64.zip
!unzip -q duckdb_cli.zip
!sudo mv duckdb /usr/local/bin/

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 59.3M  100 59.3M    0     0   199M      0 --:--:-- --:--:-- --:--:--  199M
You can now run: /usr/local/bin/aws --version
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0-:--     0
100 15.9M  100 15.9M    0     0  20.1M      0 --:--:-- --:--:-- --:--:-- 52.5M


# Download the wholde database:

In [12]:
!aws configure set region us-east-1

In [13]:
!aws s3 sync "s3://openalex" "data/oa-testing_storage/openalex-snapshot_3_11_2025" --no-sign-request

download: s3://openalex/LICENSE.txt to data/oa-testing_storage/openalex-snapshot_3_11_2025/LICENSE.txt
download: s3://openalex/RELEASE_NOTES.txt to data/oa-testing_storage/openalex-snapshot_3_11_2025/RELEASE_NOTES.txt
download: s3://openalex/README.txt to data/oa-testing_storage/openalex-snapshot_3_11_2025/README.txt
download: s3://openalex/data/authors/manifest to data/oa-testing_storage/openalex-snapshot_3_11_2025/data/authors/manifest
download: s3://openalex/data/authors/updated_date=2023-06-28/part_000.gz to data/oa-testing_storage/openalex-snapshot_3_11_2025/data/authors/updated_date=2023-06-28/part_000.gz
download: s3://openalex/browse.html to data/oa-testing_storage/openalex-snapshot_3_11_2025/browse.html
download: s3://openalex/data/authors/updated_date=2023-06-26/part_000.gz to data/oa-testing_storage/openalex-snapshot_3_11_2025/data/authors/updated_date=2023-06-26/part_000.gz
download: s3://openalex/data/authors/updated_date=2023-06-27/part_000.gz to data/oa-testing_storage/o

In [9]:
!aws s3 ls "s3://openalex" --no-sign-request


Could not connect to the endpoint URL: "https://openalex.s3.nov.amazonaws.com/?list-type=2&prefix=&delimiter=%2F&encoding-type=url"


In [10]:
!aws --version

aws-cli/2.31.27 Python/3.13.9 Linux/5.15.0-161-generic exe/x86_64.ubuntu.22


In [11]:
!aws configure list

NAME       : VALUE                    : TYPE             : LOCATION
profile    : <not set>                : None             : None
access_key : <not set>                : None             : None
secret_key : <not set>                : None             : None
region     : nov                      : imds             : 


In [14]:
import os


os.listdir('data/oa-testing_storage/openalex-snapshot_3_11_2025')

['data', 'LICENSE.txt', 'RELEASE_NOTES.txt', 'README.txt', 'browse.html']

In [4]:
!pip install numpy
!pip install pandas
!pip install matplotlib

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 50.8 MB/s  0:00:00m0:00:01
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 38.0 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 15.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 25.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━ 3/7 [fonttools]  WARNING: The scripts fonttools, pyftmerge, pyftsubset and ttx are installed in '/home/mnoichl/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [

In [5]:
import duckdb

# Connect to an existing .duckdb file (persistent storage)
con = duckdb.connect('data/oa-testing_storage/openalex-snapshot_3_11_2025/openalex.duckdb')

# Set config if you like
con.execute("SET threads = 16")
con.execute("SET memory_limit = '128GB'")

# Run a simple test query
df = con.execute("""
    SELECT id AS work_id, publication_year, cited_by_count
    FROM read_ndjson('data/oa-testing_storage/openalex-snapshot_3_11_2025/data/works/*/*.gz')
    WHERE publication_year BETWEEN 2020 AND 2024
    LIMIT 10
""").fetchdf()

df

,work_id,publication_year,cited_by_count
0,https://openalex.org/W4367174134,2023,0
1,https://openalex.org/W3127850345,2021,0
2,https://openalex.org/W3179856365,2021,0
3,https://openalex.org/W4286695661,2021,0
4,https://openalex.org/W4310902552,2022,0
5,https://openalex.org/W4287678583,2020,0
6,https://openalex.org/W3199962989,2021,0
7,https://openalex.org/W3198062397,2021,0
8,https://openalex.org/W4300082038,2021,0
9,https://openalex.org/W4312132874,2022,0


In [6]:
con.close()

In [16]:
!duckdb "data/oa-testing_storage/openalex-snapshot_3_11_2025/openalex.duckdb"

v1.1.1 af39bd0dcf
Enter ".help" for usage hints.
D 
D 
D 

In [19]:
!SET threads = 16;               -- or whatever your box has
!SET memory_limit = '128GB';      -- adjust to your allocation

/bin/bash: line 1: SET: command not found
/bin/bash: line 1: --: command not found
/bin/bash: line 1: SET: command not found
/bin/bash: line 1: --: command not found


In [8]:
import os
from pathlib import Path

def print_tree(root, prefix="", sample_head=3, sample_tail=2):
    root = Path(root)
    try:
        with os.scandir(root) as it:
            dirs, files = [], []
            for entry in it:
                (dirs if entry.is_dir(follow_symlinks=False) else files).append(entry.name)
    except PermissionError:
        print(prefix + f"[{root.name}] <permission denied>")
        return

    dirs.sort()
    files.sort()
    entries = [(True, d) for d in dirs] + [(False, f) for f in files]
    total = len(entries)

    def sampled(items):
        n = len(items)
        k = sample_head + sample_tail
        if n <= k + 1:
            return items
        return items[:sample_head] + [("sep", f"... {n - k} more ...")] + items[-sample_tail:]

    to_show = sampled(entries)

    print(prefix + f"[{root.name}]")
    for idx, item in enumerate(to_show):
        if item[0] == "sep":
            connector = "└── " if idx == len(to_show) - 1 else "├── "
            print(prefix + connector + item[1])
            continue

        is_dir, name = item
        is_last = idx == len(to_show) - 1
        connector = "└── " if is_last else "├── "
        print(prefix + connector + (f"[{name}]" if is_dir else name))

        if is_dir:
            new_prefix = prefix + ("    " if is_last else "│   ")
            print_tree(root / name, new_prefix, sample_head, sample_tail)

# Example usage
print_tree("openalex-snapshot_3_11_2025", sample_head=3, sample_tail=2)

[openalex-snapshot_3_11_2025]
├── [data]
│   [data]
│   ├── [authors]
│   │   [authors]
│   │   ├── [updated_date=2023-06-08]
│   │   │   [updated_date=2023-06-08]
│   │   │   └── part_000.gz
│   │   ├── [updated_date=2023-06-13]
│   │   │   [updated_date=2023-06-13]
│   │   │   └── part_000.gz
│   │   ├── [updated_date=2023-06-14]
│   │   │   [updated_date=2023-06-14]
│   │   │   └── part_000.gz
│   │   ├── ... 525 more ...
│   │   ├── [updated_date=2025-09-30]
│   │   │   [updated_date=2025-09-30]
│   │   │   ├── part_000.gz
│   │   │   ├── part_001.gz
│   │   │   └── part_002.gz
│   │   └── manifest
│   ├── [concepts]
│   │   [concepts]
│   │   ├── [updated_date=2023-06-07]
│   │   │   [updated_date=2023-06-07]
│   │   │   └── part_000.gz
│   │   ├── [updated_date=2023-06-08]
│   │   │   [updated_date=2023-06-08]
│   │   │   └── part_000.gz
│   │   ├── [updated_date=2024-07-09]
│   │   │   [updated_date=2024-07-09]
│   │   │   └── part_000.gz
│   │   ├── ... 24 more ...
│   │   ├── 

In [27]:
import os
from pathlib import Path

def human_size(nbytes: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB", "PB"]
    size = float(nbytes)
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.1f} {unit}"
        size /= 1024

def build_size_cache(root: Path) -> dict:
    """
    Walk the tree once and return {Path: total_size_in_bytes} for all dirs.
    Skips symlinks.
    """
    cache = {}

    def dir_size(p: Path) -> int:
        total = 0
        try:
            with os.scandir(p) as it:
                for entry in it:
                    try:
                        if entry.is_symlink():
                            continue
                        if entry.is_file(follow_symlinks=False):
                            try:
                                total += entry.stat(follow_symlinks=False).st_size
                            except OSError:
                                pass
                        elif entry.is_dir(follow_symlinks=False):
                            child = Path(entry.path)
                            total += dir_size(child)
                    except OSError:
                        continue
        except PermissionError:
            cache[p] = total
            return total
        cache[p] = total
        return total

    root = Path(root)
    if root.is_dir():
        dir_size(root)
    return cache

def print_size_tree(
    root,
    max_children=12,
    sample_head=4,
    sample_tail=3,
):
    """
    Print a tree of directories with aggregated sizes.
    If a directory has more than max_children subdirectories, only show the first
    sample_head and last sample_tail, and collapse the rest into a single "others" line.
    Files are included in each directory's total size but not listed individually.
    """
    root = Path(root)
    cache = build_size_cache(root)

    def list_subdirs(p: Path):
        try:
            with os.scandir(p) as it:
                names = [Path(e.path) for e in it if not e.is_symlink() and e.is_dir(follow_symlinks=False)]
        except PermissionError:
            names = []
        names.sort(key=lambda x: x.name.lower())
        return names

    def sampled(items):
        n = len(items)
        if n <= max_children:
            return items, [], []
        head = items[:sample_head]
        tail = items[-sample_tail:] if sample_tail else []
        omitted = items[sample_head:n - len(tail)]
        return head, tail, omitted

    def line(prefix: str, is_last: bool, text: str):
        connector = "└── " if is_last else "├── "
        print(prefix + connector + text)

    def recurse(p: Path, prefix: str = ""):
        subdirs = list_subdirs(p)
        head, tail, omitted = sampled(subdirs)

        # Header line for current directory
        size_text= human_size(cache.get(p, 0))
        print(prefix + f"[{p.name}] {size_text}")

        children = []
        if omitted:
            omitted_size = sum(cache.get(x, 0) for x in omitted)
            pseudo = f"... {len(omitted)} others ... {human_size(omitted_size)}"
            children = [("dir", d) for d in head] + [("omitted", pseudo)] + [("dir", d) for d in tail]
        else:
            children = [("dir", d) for d in subdirs]

        for i, (kind, payload) in enumerate(children):
            is_last = i == len(children) - 1
            if kind == "omitted":
                line(prefix, is_last, payload)
                continue

            child: Path = payload
            child_size = human_size(cache.get(child, 0))
            line(prefix, is_last, f"[{child.name}] {child_size}")

            new_prefix = prefix + ("    " if is_last else "│   ")
            recurse(child, new_prefix)

    if not root.exists():
        print(f"Path not found: {root}")
        return
    if not root.is_dir():
        print(f"Not a directory: {root}")
        return
    recurse(root)

# Example usage
print_size_tree("", max_children=12, sample_head=4, sample_tail=3)

[] 561.0 GB
├── [.ipynb_checkpoints] 2.0 MB
│   [.ipynb_checkpoints] 2.0 MB
├── [openalex-snapshot_3_11_2025] 422.6 GB
│   [openalex-snapshot_3_11_2025] 422.6 GB
│   └── [data] 422.6 GB
│       [data] 422.6 GB
│       ├── [authors] 57.5 GB
│       │   [authors] 57.5 GB
│       │   ├── [updated_date=2023-06-08] 1.8 KB
│       │   │   [updated_date=2023-06-08] 1.8 KB
│       │   ├── [updated_date=2023-06-13] 12.1 KB
│       │   │   [updated_date=2023-06-13] 12.1 KB
│       │   ├── [updated_date=2023-06-14] 1.7 KB
│       │   │   [updated_date=2023-06-14] 1.7 KB
│       │   ├── [updated_date=2023-06-18] 3.2 KB
│       │   │   [updated_date=2023-06-18] 3.2 KB
│       │   ├── ... 522 others ... 54.0 GB
│       │   ├── [updated_date=2025-09-28] 1.3 GB
│       │   │   [updated_date=2025-09-28] 1.3 GB
│       │   ├── [updated_date=2025-09-29] 1.4 GB
│       │   │   [updated_date=2025-09-29] 1.4 GB
│       │   └── [updated_date=2025-09-30] 869.8 MB
│       │       [updated_date=2025-09-30] 869.

In [26]:
!rm -rf value_of_philosophy_datasets/base_authors_works

In [19]:
ls

OA_setup.ipynb
openalex-snapshot_3_11_2025/
value_of_philosophy_data_pipeline.ipynb
value_of_philosophy_datasets/
value_of_philosophy_datasetsbasic_author_papers_frame.csv


In [31]:
!nproc
!free -h

15
               total        used        free      shared  buff/cache   available
Mem:           177Gi       8.9Gi       1.4Gi       1.0Mi       167Gi       167Gi
Swap:             0B          0B          0B


In [1]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:           177Gi       660Mi       176Gi       1.0Mi       497Mi       175Gi
Swap:             0B          0B          0B


In [2]:
import os
import shutil
import random
from typing import Optional


def sample_directory(
    src_root: str,
    dst_root: str,
    n_samples: int = 3,
    max_size_bytes: Optional[int] = None,
    seed: Optional[int] = None,
    follow_symlinks: bool = False,
) -> None:
    """
    Randomly sample files from a directory tree and copy them into a new tree.

    The directory structure of src_root (all subdirectories) is replicated
    under dst_root, but only up to n_samples files per directory are copied.
    Files larger than max_size_bytes can be excluded.

    Parameters
    ----------
    src_root : str
        Path to the source directory to sample from.
    dst_root : str
        Path to the destination directory where the sampled tree will be created.
    n_samples : int, optional
        Maximum number of files to copy per directory (default is 3).
    max_size_bytes : int or None, optional
        If set, files larger than this size in bytes are ignored.
        Example: 10 * 1024 * 1024 for 10 MiB. Default is None (no size limit).
    seed : int or None, optional
        Random seed for reproducibility. If None, uses non-deterministic randomness.
    follow_symlinks : bool, optional
        If True, follow symlinks to files when copying. If False, skip symlinked files.

    Returns
    -------
    None
        The function performs the copy as a side effect.

    Example
    -------
    sample_directory(
        src_root="data",
        dst_root="data_sampled",
        n_samples=3,
        max_size_bytes=10 * 1024 * 1024,  # 10 MiB
        seed=42
    )
    """
    if not os.path.isdir(src_root):
        raise ValueError(f"Source directory does not exist or is not a directory: {src_root}")

    # Prepare RNG
    rng = random.Random(seed)

    # Create the root of the destination tree
    os.makedirs(dst_root, exist_ok=True)

    for dirpath, dirnames, filenames in os.walk(src_root):
        # Compute corresponding directory under dst_root
        rel_path = os.path.relpath(dirpath, src_root)
        if rel_path == ".":
            dst_dir = dst_root
        else:
            dst_dir = os.path.join(dst_root, rel_path)

        # Make sure the directory exists in the destination
        os.makedirs(dst_dir, exist_ok=True)

        # Filter candidate files based on size and symlink rules
        candidates = []
        for fname in filenames:
            src_path = os.path.join(dirpath, fname)

            # Skip if it is not a regular file
            # (for safety, we check it before size and symlink logic)
            if not os.path.isfile(src_path):
                continue

            # Handle symlinks
            if not follow_symlinks and os.path.islink(src_path):
                continue

            # Check size limit if specified
            if max_size_bytes is not None:
                try:
                    size = os.path.getsize(src_path)
                except OSError:
                    # If we cannot get size, skip this file
                    continue
                if size > max_size_bytes:
                    continue

            candidates.append(src_path)

        # If there are fewer candidates than requested, just take all of them
        if len(candidates) <= n_samples:
            selected = candidates
        else:
            selected = rng.sample(candidates, n_samples)

        # Copy selected files
        for src_path in selected:
            fname = os.path.basename(src_path)
            dst_path = os.path.join(dst_dir, fname)

            # Perform the copy (preserve metadata)
            shutil.copy2(src_path, dst_path)



sample_directory(
    src_root="openalex-snapshot_3_11_2025",
    dst_root="openalex-snapshot_3_11_2025_sampled",
    n_samples=3,
    max_size_bytes=50 * 1024 * 1024,  # 50 MiB
    seed=123
)

In [6]:
import os
import shutil
import random
import re
from typing import Optional


def sample_directory(
    src_root: str,
    dst_root: str,
    n_samples: int = 3,
    max_size_bytes: Optional[int] = None,
    seed: Optional[int] = None,
    follow_symlinks: bool = False,
    dir_sample_regex: Optional[str] = None,
    dir_samples_per_level: Optional[int] = None,
) -> None:
    """
    Randomly sample files from a directory tree and copy them into a new tree.

    The directory structure of src_root is replicated under dst_root, but only
    up to n_samples files per directory are copied.

    You can also subsample folders on each level that match a given pattern
    for example directories named "updated_date=2023-05-29", "updated_date=2023-05-30", etc.

    Parameters
    ----------
    src_root : str
        Source directory.
    dst_root : str
        Destination directory where the sampled tree will be created.
    n_samples : int
        Maximum number of files to copy per directory.
    max_size_bytes : int or None
        If set, files larger than this size (in bytes) are ignored.
    seed : int or None
        Random seed for reproducibility.
    follow_symlinks : bool
        If True, copy the content of symlinked files. If False, skip symlinked files.
    dir_sample_regex : str or None
        Regular expression applied to directory names. Only directories whose
        name matches this regex are candidates for subsampling. Others are always kept.
        Example: r"^updated_date=" to match folders like "updated_date=2023-05-29".
    dir_samples_per_level : int or None
        If set and dir_sample_regex is not None, then on each directory level
        at most this many matching subdirectories are traversed and copied.
        If there are fewer than this number, all matching ones are kept.

    Returns
    -------
    None
    """
    if not os.path.isdir(src_root):
        raise ValueError(f"Source directory does not exist or is not a directory: {src_root}")

    rng = random.Random(seed)
    os.makedirs(dst_root, exist_ok=True)

    dir_pattern = re.compile(dir_sample_regex) if dir_sample_regex is not None else None

    # Use topdown so that pruning dirnames affects the walk
    for dirpath, dirnames, filenames in os.walk(src_root, topdown=True):
        # Optionally subsample directories on this level
        if dir_pattern is not None and dir_samples_per_level is not None:
            matching = [d for d in dirnames if dir_pattern.search(d)]
            if len(matching) > dir_samples_per_level:
                keep_matching = rng.sample(matching, dir_samples_per_level)
            else:
                keep_matching = matching

            non_matching = [d for d in dirnames if d not in matching]
            # Update dirnames in place so os.walk only descends into the kept ones
            dirnames[:] = non_matching + keep_matching

        # Compute destination directory path
        rel_path = os.path.relpath(dirpath, src_root)
        dst_dir = dst_root if rel_path == "." else os.path.join(dst_root, rel_path)
        os.makedirs(dst_dir, exist_ok=True)

        # Filter candidate files
        candidates = []
        for fname in filenames:
            src_path = os.path.join(dirpath, fname)

            if not os.path.isfile(src_path):
                continue

            if not follow_symlinks and os.path.islink(src_path):
                continue

            if max_size_bytes is not None:
                try:
                    size = os.path.getsize(src_path)
                except OSError:
                    continue
                if size > max_size_bytes:
                    continue

            candidates.append(src_path)

        # Sample files
        if len(candidates) <= n_samples:
            selected = candidates
        else:
            selected = rng.sample(candidates, n_samples)

        # Copy selected files
        for src_path in selected:
            fname = os.path.basename(src_path)
            dst_path = os.path.join(dst_dir, fname)
            shutil.copy2(src_path, dst_path)


if __name__ == "__main__":
    # Example for a tree like the screenshot:
    #   root/
    #     updated_date=2023-05-29/
    #     updated_date=2023-05-30/
    #     ...
    #
    # This keeps at most 5 "updated_date=..." folders per level,
    # and within each copied folder keeps at most 3 files per directory.
    sample_directory(
         src_root="openalex-snapshot_3_11_2025",
        dst_root="openalex-snapshot_3_11_2025_sampled_",
        n_samples=3,
        max_size_bytes=1500 * 1024 * 1024,
        seed=123,
        dir_sample_regex=r"^updated_date=",
        dir_samples_per_level=3,
    )

In [5]:
!rm -rf openalex-snapshot_3_11_2025_sampled

In [13]:
!zip -r snapshot_dev_sample.zip openalex-snapshot_3_11_2025_sampled_

  adding: openalex-snapshot_3_11_2025_sampled_/ (stored 0%)
  adding: openalex-snapshot_3_11_2025_sampled_/LICENSE.txt (deflated 60%)
  adding: openalex-snapshot_3_11_2025_sampled_/README.txt (deflated 10%)
  adding: openalex-snapshot_3_11_2025_sampled_/openalex.duckdb (deflated 99%)
  adding: openalex-snapshot_3_11_2025_sampled_/data/ (stored 0%)
  adding: openalex-snapshot_3_11_2025_sampled_/data/authors/ (stored 0%)
  adding: openalex-snapshot_3_11_2025_sampled_/data/authors/manifest (deflated 90%)
  adding: openalex-snapshot_3_11_2025_sampled_/data/authors/updated_date=2025-06-13/ (stored 0%)
  adding: openalex-snapshot_3_11_2025_sampled_/data/authors/updated_date=2025-06-13/part_000.gz (deflated 0%)
  adding: openalex-snapshot_3_11_2025_sampled_/data/authors/updated_date=2025-01-13/ (stored 0%)
  adding: openalex-snapshot_3_11_2025_sampled_/data/authors/updated_date=2025-01-13/part_000.gz (deflated 0%)
  adding: openalex-snapshot_3_11_2025_sampled_/data/authors/updated_date=2024-0

In [12]:
!sudo apt install zip unzip -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unzip is already the newest version (6.0-26ubuntu3.2).
The following NEW packages will be installed:
  zip
0 upgraded, 1 newly installed, 0 to remove and 1 not upgraded.
Need to get 176 kB of archives.
After this operation, 544 kB of additional disk space will be used.
Get:1 http://nova.clouds.archive.ubuntu.com/ubuntu jammy/main amd64 zip amd64 3.0-12build2 [176 kB]
Fetched 176 kB in 1s (292 kB/s)m

78Selecting previously unselected package zip.
(Reading database ... 106953 files and directories currently installed.)
Preparing to unpack .../zip_3.0-12build2_amd64.deb ...
7Progress: [  0%] [..........................................................] 87Progress: [ 20%] [###########...............................................] 8Unpacking zip (3.0-12build2) ...
7Progress: [ 40%] [#######################...................................] 8Setting up zip (3.0-12build2) ...
7Progres